In [30]:
import torch
import numpy as np
from matplotlib import pyplot as plt

import math
import random

In [5]:
PATHS = {
    "train": "../data/UCI-HAR/train/",
    "test": "../data/UCI-HAR/test/"
}
NUM_CONCEPTS = 2
WINDOW_TIMESTEPS = 128

In [ ]:
tot_acc_x = np.loadtxt("{0}Inertial Signals/total_acc_x_{1}.txt".format(PATHS["train"], "train"))
activities = np.loadtxt("{0}y_{1}.txt".format(PATHS["train"], "train"))

cv_values = list()

for idx in range(0, len(activities)):
    # We skip static activities and only consider dynamic ones
    if activities[idx] > 3:
        continue

    peaks = list()
    for timestep in range(1, WINDOW_TIMESTEPS - 1):
        # We look for peaks in the time window of the sample
        if tot_acc_x[idx][timestep - 1] < tot_acc_x[idx][timestep] and tot_acc_x[idx][timestep + 1] < tot_acc_x[idx][timestep]:
            peaks.append(tot_acc_x[idx][timestep])

    mean = sum(peaks) / len(peaks)
    var = 0

    for peak in peaks:
        var += (peak - mean)**2
    var = var / len(peaks)

    std = math.sqrt(var)
    cv = (std / mean).item()

    cv_values.append(cv)

print(cv_values)


# K-means algorithm
cv_tensor = torch.tensor(cv_values)
centroids = torch.tensor(random.sample(cv_values, 2))
print(centroids)

num_iterations = 100

for _ in range(0, num_iterations):
    labels = torch.empty(cv_tensor.shape)
    for i in range(0, len(labels)):
        if abs(cv_tensor[i] - centroids[0]) < abs(cv_tensor[i] - centroids[1]):
            labels[i] = 0
        else:
            labels[i] = 1

    for i in range(0, 4):
        if torch.sum(labels == i) > 0:
            centroids[i] = torch.mean(cv_tensor[labels == i], dim=0)

print(centroids)

[0.18133903926747238, 0.21894812662596413, 0.21124325772170807, 0.13908603585640064, 0.2325729213863667, 0.2717715996608985, 0.22777852666823725, 0.17638092782652426, 0.18987832017564146, 0.22507845223251421, 0.24693216173253146, 0.23840156426877068, 0.24571513444268298, 0.2616232063216425, 0.2628001525156302, 0.22603889700429483, 0.18272118114375313, 0.14164945359713407, 0.16820824645307839, 0.16199901995215038, 0.20461817864109966, 0.27475069878915603, 0.2295144506558699, 0.1791008671699646, 0.2068732428491681, 0.21472920788780794, 0.23894149504073586, 0.25648725481825463, 0.2205894801093642, 0.2395622306861138, 0.21720843156600397, 0.2107288279021992, 0.21342898217241058, 0.22971516221408078, 0.27411032863887014, 0.2764130291797902, 0.2511512634346118, 0.22596336384740714, 0.18165543932251116, 0.16311609689585416, 0.15997377879811178, 0.17618626643024746, 0.16481645634056918, 0.159306573119301, 0.19161888460939722, 0.1931289335890099, 0.19712707557207457, 0.281091725167799, 0.307836

In [ ]:
def clear_concepts (type):
    # Clearing the previous content of the concept files (if any)
    concepts_file = open("{0}concepts_{1}.txt".format(PATHS[type], type), "w")
    concepts_file.write("")
    concepts_file.close()


def concept_labeling (type):
    concepts_file = open("{0}concepts_{1}.txt".format(PATHS[type], type), "a")

    activities = np.loadtxt("{0}y_{1}.txt".format(PATHS[type], type))
    tot_acc_x = np.loadtxt("{0}Inertial Signals/total_acc_x_{1}.txt".format(PATHS[type], type))
    tot_acc_y = np.loadtxt("{0}Inertial Signals/total_acc_y_{1}.txt".format(PATHS[type], type))
    tot_acc_z = np.loadtxt("{0}Inertial Signals/total_acc_z_{1}.txt".format(PATHS[type], type))

    concepts = np.empty((len(activities), NUM_CONCEPTS))

    for idx in range(0, len(concepts)):
        # Labeling dynamic (1) or static (0) for classes [1, 3] and [4, 6]
        if int(activities[idx]) <= 3:
            concepts[idx][0] = 1
        else:
            concepts[idx][0] = 0

        # Labeling horizontal posture (0, lying) or vertical posture (1, all the others) 
        x_sum = 0
        y_sum = 0
        z_sum = 0
        for timestep in range(0, WINDOW_TIMESTEPS):
            x_sum += tot_acc_x[idx][timestep]
            y_sum += tot_acc_y[idx][timestep]
            z_sum += tot_acc_z[idx][timestep]

        if abs(y_sum) < abs(x_sum) and abs(z_sum) < abs(x_sum):
            concepts[idx][1] = 1
        else:
            concepts[idx][1] = 0

        # Labeling the regularity of the 
        


    # The calculated concepts are written on the related txt file
    for idx in range(0, len(concepts)):
        for concept in concepts[idx]:
            concepts_file.write(str(concept) + "  ")
        concepts_file.write("\n")

    concepts_file.close()

    

In [8]:
clear_concepts("train")
clear_concepts("test")

concept_labeling("train")
concept_labeling("test")